# Phase 1 Pipeline Check

This notebook verifies the end-to-end extraction and core feature building pipeline over the master dataset.
It implements exact instructions scoped under DiscoveryRank without leaking ranking logic too early.

In [ ]:
import sys
import os
import pandas as pd

# Add src dynamically
sys.path.append(os.path.abspath('../src'))
import data_prep
import session_builder
import relevance_labels
import freshness_features

In [ ]:
data_dir = '../data'
log_files = ['log_random_4_22_to_5_08_1k.csv']  # Sample using single random file to execute quickly
sample_size = 50000

print("1. Data Prep...")
df = data_prep.load_and_merge_data(data_dir, log_files, sample_size=sample_size)
print(f"Shape after merge: {df.shape}")
print(f"Columns present: {df.columns.tolist()}\n")

print("2. Session Builder...")
df = session_builder.assign_sessions(df)
print(f"Unique sessions created over {df['user_id'].nunique()} users: {df['session_id'].nunique()}\n")

print("3. Relevance Labels...")
df = relevance_labels.create_relevance_labels(df)
print("Relevance Target (y_relevant) Distribution:")
print(df['y_relevant'].value_counts(normalize=True).to_dict())
print("\n")

print("4. Freshness Features...")
df = freshness_features.calculate_freshness(df)
print("Item Age summary (days):")
print(df['item_age_days'].describe())

print("\n5. Saving output...")
output_path = '../outputs/pipeline_sample.csv'
df.to_csv(output_path, index=False)
print(f"Saved final pipeline df successfully to {output_path}!\n")

df.head()